# Pathfinding Visual Test

Interactive 3D visualisation of the recursive midpoint-lifting pathfinder.
Run all cells to see **clear** vs **obstructed** scenarios side by side,
then tweak the custom scenario at the bottom.

In [14]:
%matplotlib tk

import numpy as np
import trimesh
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.widgets import Slider
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from src.pathfinding import find_path

OBJECTS_DIR = Path("../my_simulation/3d_objects")


def plot_path_3d(ax, mesh, path, A, B):
    """Draw the mesh, start/end points, and computed path on a 3D axes."""
    triangles = mesh.vertices[mesh.faces]
    poly = Poly3DCollection(triangles, alpha=0.25, facecolor="grey", edgecolor="grey", linewidth=0.2)
    ax.add_collection3d(poly)

    ax.scatter(*A, color="green", s=60, zorder=5, label="A (start)")
    ax.scatter(*B, color="red", s=60, zorder=5, label="B (end)")

    pts = np.array(path)
    ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color="blue", linewidth=2, zorder=4)
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], color="blue", s=20, zorder=4, label="waypoints")

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.legend(loc="upper left", fontsize=7)

In [15]:
# Load and scale meshes to ~10 cm
duck = trimesh.load(str(OBJECTS_DIR / "duckround.stl"), force="mesh")
duck.apply_transform(np.diag([0.10 / duck.extents.max()] * 3 + [1.0]))

trapez = trimesh.load(str(OBJECTS_DIR / "trapez.stl"), force="mesh")
trapez.apply_transform(np.diag([0.10 / trapez.extents.max()] * 3 + [1.0]))

print(f"Duck   extents: {np.round(duck.extents*100, 1)} cm")
print(f"Trapez extents: {np.round(trapez.extents*100, 1)} cm")

# Compute paths
path_duck_clear = find_path(np.array([-0.15, 0.0, 0.10]), np.array([0.15, 0.0, 0.10]), duck)
path_duck_obst  = find_path(np.array([-0.15, 0.0, 0.03]), np.array([0.15, 0.0, 0.03]), duck)
path_trap_obst  = find_path(np.array([-0.08, 0.0, 0.01]), np.array([0.08, 0.0, 0.01]), trapez)

Duck   extents: [10.   7.4  7.8] cm
Trapez extents: [10.   6.7  3.8] cm


In [16]:
scenes = [
    ("Duck — clear path",      duck,   path_duck_clear, np.array([-0.15, 0.0, 0.10]), np.array([0.15, 0.0, 0.10])),
    ("Duck — obstructed path", duck,   path_duck_obst,  np.array([-0.15, 0.0, 0.03]), np.array([0.15, 0.0, 0.03])),
    ("Trapez — obstructed",    trapez, path_trap_obst,  np.array([-0.08, 0.0, 0.01]), np.array([0.08, 0.0, 0.01])),
]

fig = plt.figure(figsize=(18, 7))
fig.subplots_adjust(bottom=0.15)
axes = [fig.add_subplot(1, 3, i + 1, projection="3d") for i in range(3)]

for ax, (title, mesh, path, A, B) in zip(axes, scenes):
    ax.set_title(f"{title} ({len(path)} pts)")
    plot_path_3d(ax, mesh, path, A, B)
    ax.view_init(elev=25, azim=-60)
    ax.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.06, 0.6, 0.03])
azim_slider = Slider(azim_ax, "Azimuth", -180, 180, valinit=-60, valstep=1)

elev_ax = fig.add_axes([0.2, 0.02, 0.6, 0.03])
elev_slider = Slider(elev_ax, "Elevation", -90, 90, valinit=25, valstep=1)

def update_view(val):
    for ax in axes:
        ax.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view)
elev_slider.on_changed(update_view)

plt.show()

In [17]:
# Custom scenario — pick a mesh and edit A / B, then re-run
mesh = duck  # or trapez
A_custom = np.array([-0.10, 0.0, 0.01])
B_custom = np.array([ 0.10, 0.0, 0.01])

path_custom = find_path(A_custom, B_custom, mesh)
print(f"Path has {len(path_custom)} waypoints")

fig2 = plt.figure(figsize=(10, 8))
fig2.subplots_adjust(bottom=0.15)
ax2 = fig2.add_subplot(111, projection="3d")
ax2.set_title("Custom scenario")
plot_path_3d(ax2, mesh, path_custom, A_custom, B_custom)
ax2.view_init(elev=25, azim=-60)
ax2.disable_mouse_rotation()

azim_ax2 = fig2.add_axes([0.2, 0.06, 0.6, 0.03])
azim_slider2 = Slider(azim_ax2, "Azimuth", -180, 180, valinit=-60, valstep=1)

elev_ax2 = fig2.add_axes([0.2, 0.02, 0.6, 0.03])
elev_slider2 = Slider(elev_ax2, "Elevation", -90, 90, valinit=25, valstep=1)

def update_custom(val):
    ax2.view_init(elev=elev_slider2.val, azim=azim_slider2.val)
    fig2.canvas.draw_idle()

azim_slider2.on_changed(update_custom)
elev_slider2.on_changed(update_custom)

plt.show()

Path has 3 waypoints
